## Section 2.0 — Preamble

**What this does**
- Introduces Feature Engineering stage
- Provides context before payload TF-IDF

**Why**
- Maintain consistency with Style Guide
- Ensure readers know scope of Section 2


In [1]:
# ======================================================
# Section 2.0 — Preamble
#   • Introduces Feature Engineering stage
#   • Provides context before payload TF-IDF
# ======================================================
print(">>> Section 2.0: Feature Engineering Preamble ready")


>>> Section 2.0: Feature Engineering Preamble ready


## Section 2.1 — Payload Sequence (TF-IDF)

**What this does**
- Folds `payload_byte_*` into single `payload` sequence
- Applies TF-IDF vectorisation for n-gram features

**Why**
- Converts raw payload bytes into sparse numeric features
- Captures frequency patterns for anomaly detection


In [2]:
# ======================================================
# Section 2.1 — Payload Sequence (TF-IDF)
#   • Fold payload bytes into single sequence
#   • Apply TF-IDF vectorisation
# ======================================================
print(">>> Section 2.1: Payload Sequence TF-IDF started")

# Placeholder: implement folding + TfidfVectorizer pipeline here
# Example skeleton:
# from sklearn.feature_extraction.text import TfidfVectorizer
# vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4), max_features=5000)
# X_payload = vectorizer.fit_transform(df['payload'].astype(str))
# print('TF-IDF payload shape:', X_payload.shape)


>>> Section 2.1: Payload Sequence TF-IDF started



# 02 — Feature Engineering (Patched)

This patched notebook:
- Audits & normalizes the binary target to `__target__` (0/1).
- Builds stratified Train/Val/Test with **at least one positive in val/test when possible**.
- Engineers structured features (scaled numeric + one-hot categorical).
- Optionally adds TF‑IDF features from text-like columns.
- Saves staged arrays for downstream notebooks:
  - `X_train.npy`, `X_val.npy`, `X_test.npy` (structured only, CSR)
  - `X_train_all.npy`, `X_val_all.npy`, `X_test_all.npy` (structured + text, CSR when text exists)
  - `y_train.npy`, `y_val.npy`, `y_test.npy`
  - `02_feature_meta.json`


In [3]:

# ==== Prologue & configuration ====
from pathlib import Path
import os, json, warnings, gc

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_STATE = globals().get("RANDOM_STATE", 42)
OUT_ROOT = Path(globals().get("OUT_ROOT", "out"))
STAGE_ROOT = Path(globals().get("STAGE_ROOT", OUT_ROOT))
SPLIT_DIR = STAGE_ROOT / "split_preproc"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[02] RANDOM_STATE={RANDOM_STATE}")
print(f"[02] OUT_ROOT={OUT_ROOT}  STAGE_ROOT={STAGE_ROOT}")
print(f"[02] Will write staged arrays to: {SPLIT_DIR}")


[02] RANDOM_STATE=42
[02] OUT_ROOT=out  STAGE_ROOT=out
[02] Will write staged arrays to: out/split_preproc


In [4]:
# ======================================================
# [02] Robust Data Ingestion + Label Fix (drop-in cell)
# ======================================================
print(">>> [02] Data Ingestion (robust)")

import os, glob, numpy as np, pandas as pd
from pathlib import Path

# Defaults if 01 didn't run yet (harmless if already set)
OUT_ROOT = globals().get("OUT_ROOT", "out")
STAGE_ROOT = globals().get("STAGE_ROOT", "out/stage")
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(STAGE_ROOT).mkdir(parents=True, exist_ok=True)
reports = Path(OUT_ROOT) / "reports"; reports.mkdir(parents=True, exist_ok=True)

def _read_any(p: Path) -> pd.DataFrame:
    return pd.read_csv(p) if p.suffix.lower()==".csv" else pd.read_parquet(p)

# 1) If a DataFrame `df` already exists, use it
if "df" in globals() and isinstance(df, pd.DataFrame) and len(df) > 0:
    data = df.copy()
    source = "in-memory df"
else:
    # 2) Try to auto-discover a dataset
    proj = Path.cwd()
    cands = []

    # env hints
    for key in ("DATA_PATH", "RAW_PATH", "RAW_ROOT"):
        p = os.environ.get(key)
        if p:
            pth = Path(p)
            if pth.is_file():
                cands.append(pth)
            else:
                cands += [Path(x) for x in glob.glob(str(pth / "*.csv"))]
                cands += [Path(x) for x in glob.glob(str(pth / "*.parquet"))]

    # common relative locations
    rel_dirs = ["data", "data/raw", "dataset", "datasets", "input", "inputs"]
    for r in rel_dirs:
        cands += [Path(x) for x in glob.glob(str(proj / r / "*.csv"))]
        cands += [Path(x) for x in glob.glob(str(proj / r / "*.parquet"))]

    # root-level files
    cands += [Path(x) for x in glob.glob(str(proj / "*.csv"))]
    cands += [Path(x) for x in glob.glob(str(proj / "*.parquet"))]

    cands = [p for p in cands if p.is_file()]
    if cands:
        best = max(cands, key=lambda p: p.stat().st_size)
        print(f"[02] Found dataset → {best} ({best.stat().st_size/1e6:.2f} MB)")
        data = _read_any(best)
        source = str(best)
    else:
        # 3) No data found → synth fallback with positives
        print("[02] No dataset found; generating synthetic data (~5% positives).")
        n = 60000
        rng = np.random.default_rng(42)
        Xnum = rng.normal(size=(n, 8))
        Xcat = rng.integers(0, 5, size=(n, 3)).astype(str)
        score = 0.6*Xnum[:,0] - 0.4*Xnum[:,1] + 0.2*rng.normal(size=n)
        y = (score > np.quantile(score, 0.95)).astype(int)  # ~5% positives
        data = pd.DataFrame(
            {**{f"num_{i}": Xnum[:, i] for i in range(8)},
             **{f"cat_{i}": Xcat[:, i] for i in range(3)},
             "__target__": y}
        )
        synth_dir = Path(OUT_ROOT) / "synthetic"; synth_dir.mkdir(parents=True, exist_ok=True)
        data.to_csv(synth_dir / "synthetic_df.csv", index=False)
        (reports / "02_synthetic_note.txt").write_text(
            "No dataset found; generated synthetic DataFrame with ~5% positives.\n"
        )
        source = "synthetic_fallback"

# 4) Clean index
data = data.reset_index(drop=True)

# 5) Ensure a binary target named __target__
target_candidates = ["__target__", "target", "label", "y", "class", "fraud", "churn", "is_bad"]
tcol = next((c for c in target_candidates if c in data.columns), None)

if tcol is None:
    # Create label from first numeric feature (top 5% as positives) if none present
    num_cols = [c for c in data.columns if pd.api.types.is_numeric_dtype(data[c])]
    if len(num_cols) == 0:
        data["feat_const"] = 1.0
        data["__target__"] = 0
    else:
        s = pd.to_numeric(data[num_cols[0]], errors="coerce").fillna(0.0)
        thr = s.quantile(0.95)
        data["__target__"] = (s > thr).astype(int)
else:
    if tcol != "__target__":
        data["__target__"] = data[tcol]

# Coerce to clean 0/1
data["__target__"] = (
    pd.Series(data["__target__"])
      .map({True: 1, False: 0, "1": 1, "0": 0})
      .astype("Int64")
      .astype("float")
      .fillna(0)
      .astype(int)
)

# If still all zeros → surrogate label from a numeric feature
if int(data["__target__"].sum()) == 0:
    num_cols = [c for c in data.columns if c != "__target__" and pd.api.types.is_numeric_dtype(data[c])]
    if num_cols:
        s = pd.to_numeric(data[num_cols[0]], errors="coerce").fillna(0.0)
    else:
        # as a last resort, random normal score
        s = pd.Series(np.random.default_rng(0).normal(size=len(data)))
    q = 0.98 if len(data) > 10000 else 0.90
    thr = s.quantile(q)
    data["__target__"] = (s > thr).astype(int)
    (reports / "02_label_fix_note.txt").write_text(
        "Original label had no positives; created surrogate label via top-quantile of a numeric feature.\n"
    )
    print("[02] Label was all-zero; created surrogate positives via numeric quantile threshold.")

print(f"[02] df ready: shape={data.shape}, positives={int(data.__target__.sum())} "
      f"({data.__target__.mean()*100:.2f}%), source={source}")
df = data


>>> [02] Data Ingestion (robust)
[02] No dataset found; generating synthetic data (~5% positives).


[02] df ready: shape=(60000, 12), positives=3000 (5.00%), source=synthetic_fallback


In [5]:

# ==== Data loading ====
# If a DataFrame `df` already exists in memory, use it.
# Otherwise try common locations.
def _try_read():
    candidates = [
        "data/processed.parquet",
        "data/processed.parq",
        "data/processed.feather",
        "data/processed.csv",
        "data/raw.parquet",
        "data/raw.csv",
        "out/dataset.parquet",
        "out/dataset.csv",
    ]
    for p in candidates:
        fp = Path(p)
        if fp.exists():
            try:
                if fp.suffix in (".parquet",".parq"):
                    return pd.read_parquet(fp)
                elif fp.suffix == ".feather":
                    return pd.read_feather(fp)
                else:
                    return pd.read_csv(fp)
            except Exception as e:
                print(f"[02] Failed reading {fp}: {e}")
    return None

if "df" in globals() and isinstance(df, pd.DataFrame):
    data = df.copy()
    print("[02] Using in-memory DataFrame `df` (copy).")
else:
    data = _try_read()
    assert data is not None, "[02] No DataFrame `df` in memory and no dataset found in common paths."
    print(f"[02] Loaded dataset shape={data.shape}")

# Ensure row index is clean
data = data.reset_index(drop=True)
print(f"[02] Columns: {list(data.columns)[:10]}{'...' if data.shape[1]>10 else ''}")


[02] Using in-memory DataFrame `df` (copy).
[02] Columns: ['num_0', 'num_1', 'num_2', 'num_3', 'num_4', 'num_5', 'num_6', 'num_7', 'cat_0', 'cat_1']...


In [6]:

# ==== Target detection & normalization ====
# Prefer an explicit TARGET_COL, else auto-detect a binary-like column.

TARGET_COL = globals().get("TARGET_COL", None)

def _is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    if len(vals) == 0:
        return False
    # numeric or boolean or typical strings
    try:
        # coerce stringy bools
        tmp = pd.Series(s).astype(str).str.lower().str.strip()
        mapped = tmp.map({"true":1,"false":0,"yes":1,"no":0,"y":1,"n":0,"t":1,"f":0})
        # fallback: numeric coercion
        if mapped.isna().all():
            mapped = pd.to_numeric(tmp, errors="coerce")
        u = pd.Series(mapped.dropna().unique())
        return set(u.dropna().astype(int).unique().tolist()).issubset({0,1}) and len(u) <= 2
    except Exception:
        return False

def _coerce_to01(s):
    tmp = pd.Series(s)
    # try booleans / yes-no / strings
    s_norm = (
        tmp.astype(str).str.lower().str.strip()
        .map({"true":1,"false":0,"yes":1,"no":0,"y":1,"n":0,"t":1,"f":0})
    )
    if s_norm.isna().all():
        s_norm = pd.to_numeric(tmp, errors="coerce")
    # If still not {0,1}, binarize on most common value as 0, others as 1 (last resort)
    u = s_norm.dropna().unique()
    if not set(pd.Series(u).dropna().astype(int).unique().tolist()).issubset({0,1}):
        # fallback: mode -> 0, else 1
        mode_val = s_norm.mode(dropna=True)
        mode_val = mode_val.iloc[0] if not mode_val.empty else 0
        s_norm = s_norm.fillna(mode_val)
        s_norm = (s_norm != mode_val).astype(int)
    return s_norm.fillna(0).astype(int)

if TARGET_COL is None or TARGET_COL not in data.columns:
    # common candidate names
    candidates = [
        "target","label","y","outcome","is_fraud","fraud","churn","clicked","response","default","is_positive"
    ]
    # any binary-looking columns
    bin_cols = [c for c in data.columns if _is_binary_series(data[c])]
    pick = None
    for c in candidates:
        if c in bin_cols:
            pick = c; break
    if pick is None and len(bin_cols) > 0:
        pick = bin_cols[0]
    assert pick is not None, "[02] Could not auto-detect a binary target column. Set TARGET_COL before running."
    TARGET_COL = pick
    print(f"[02] Auto-detected target column: {TARGET_COL}")
else:
    print(f"[02] Using provided TARGET_COL: {TARGET_COL}")

data["__target__"] = _coerce_to01(data[TARGET_COL])
# Drop rows with missing target after coercion (unlikely)
mask = data["__target__"].isin([0,1])
dropped = (~mask).sum()
if dropped:
    print(f"[02] Dropping {dropped} rows with invalid target values.")
data = data.loc[mask].reset_index(drop=True)

y = data["__target__"].to_numpy(dtype=int)
pos_total = int((y==1).sum())
neg_total = int((y==0).sum())
print(f"[02] Target balance total: pos={pos_total}, neg={neg_total} (n={len(y)})")


[02] Auto-detected target column: __target__
[02] Target balance total: pos=3000, neg=57000 (n=60000)


In [7]:

# ==== Train / Val / Test split with min positives in val/test when possible ====
VAL_RATIO  = float(globals().get("VAL_RATIO", 0.2))
TEST_RATIO = float(globals().get("TEST_RATIO", 0.2))

assert 0.0 < VAL_RATIO < 0.9 and 0.0 < TEST_RATIO < 0.9 and VAL_RATIO+TEST_RATIO < 0.9, \
    "[02] VAL_RATIO/TEST_RATIO must be reasonable."

def _minpos_split(y, val_ratio, test_ratio, random_state=42):
    n = len(y)
    idx = np.arange(n)
    pos_idx = idx[y==1]
    neg_idx = idx[y==0]
    rng = np.random.default_rng(random_state)

    # Shuffle
    rng.shuffle(pos_idx); rng.shuffle(neg_idx)

    n_val  = int(round(n * val_ratio))
    n_test = int(round(n * test_ratio))
    n_train = n - n_val - n_test

    # Ensure at least one positive in val & test when possible
    p = len(pos_idx)
    need_val_pos = 1 if p >= 1 else 0
    need_test_pos = 1 if p >= 2 else (1 if p>=1 else 0)  # if only 1 positive, allocate to val

    val_pos = pos_idx[:need_val_pos]
    test_pos = pos_idx[need_val_pos:need_val_pos+need_test_pos]
    rem_pos = pos_idx[need_val_pos+need_test_pos:]

    # Fill remaining slots with negatives / remaining positives proportional to sizes
    val_need = n_val - len(val_pos)
    test_need = n_test - len(test_pos)

    # combine remaining pool
    pool = np.concatenate([rem_pos, neg_idx])
    rng.shuffle(pool)
    val_extra = pool[:max(0,val_need)]
    test_extra = pool[max(0,val_need):max(0,val_need)+max(0,test_need)]
    train_rest = pool[max(0,val_need)+max(0,test_need):]

    val_idx = np.concatenate([val_pos, val_extra])
    test_idx = np.concatenate([test_pos, test_extra])

    # Remainder to train + any leftovers
    # Also include any neg left unused at the top (but we've used them via pool)
    used = set(val_idx.tolist() + test_idx.tolist())
    train_idx = np.array([i for i in idx if i not in used])

    # Safety: sizes
    # Adjust if off by rounding
    if len(train_idx) + len(val_idx) + len(test_idx) != n:
        # fallback to simple split
        X_tr, X_tmp, y_tr, y_tmp, idx_tr, idx_tmp = train_test_split(
            np.zeros((n,1)), y, idx, test_size=(val_ratio+test_ratio), stratify=y if len(np.unique(y))>1 else None, random_state=random_state
        )
        X_va, X_te, y_va, y_te, idx_va, idx_te = train_test_split(
            X_tmp, y_tmp, idx_tmp, test_size=(test_ratio/(val_ratio+test_ratio)), stratify=y_tmp if len(np.unique(y_tmp))>1 else None, random_state=random_state
        )
        return idx_tr, idx_va, idx_te

    return train_idx, val_idx, test_idx

if len(np.unique(y))>1:
    tr_idx, va_idx, te_idx = _minpos_split(y, VAL_RATIO, TEST_RATIO, RANDOM_STATE)
else:
    # Single-class dataset: keep simple random split (no-stratify possible)
    idx = np.arange(len(y))
    tr_idx, tmp_idx = train_test_split(idx, test_size=(VAL_RATIO+TEST_RATIO), random_state=RANDOM_STATE, shuffle=True)
    va_idx, te_idx = train_test_split(tmp_idx, test_size=(TEST_RATIO/(VAL_RATIO+TEST_RATIO)), random_state=RANDOM_STATE, shuffle=True)

def _split_df(df, tr_idx, va_idx, te_idx):
    tr = df.iloc[tr_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    te = df.iloc[te_idx].reset_index(drop=True)
    return tr, va, te

train_df, val_df, test_df = _split_df(data, tr_idx, va_idx, te_idx)
print("[02] Split sizes:", len(train_df), len(val_df), len(test_df))

def _counts(s):
    u, c = np.unique(s, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))

print("[02] y_train counts:", _counts(train_df["__target__"].values))
print("[02] y_val   counts:", _counts(val_df["__target__"].values))
print("[02] y_test  counts:", _counts(test_df["__target__"].values))


[02] Split sizes: 36000 12000 12000
[02] y_train counts: {0: 34173, 1: 1827}
[02] y_val   counts: {0: 11406, 1: 594}
[02] y_test  counts: {0: 11421, 1: 579}


In [8]:

# ==== Feature groups ====
# Identify numeric, categorical, and text-like columns.
# Drop id-like or target columns from features.

def _is_texty_series(s, max_unique_ratio=0.95):
    if s.dtype == object:
        # text-like if average string length > 5 or high cardinality
        sample = pd.Series(s.dropna().astype(str).head(1000))
        if sample.empty:
            return False
        avg_len = sample.str.len().mean()
        nunique = s.nunique(dropna=True)
        return (avg_len >= 5) or (nunique / max(1, len(s)) > max_unique_ratio)
    return False

skip_cols = set(["__target__", TARGET_COL])
# Common id-like patterns
skip_cols |= {c for c in data.columns if c.lower() in ("id","uuid","guid","index")}

feature_cols = [c for c in data.columns if c not in skip_cols]

num_cols = []
cat_cols = []
txt_cols = []

for c in feature_cols:
    s = data[c]
    if pd.api.types.is_numeric_dtype(s):
        num_cols.append(c)
    elif _is_texty_series(s):
        txt_cols.append(c)
    else:
        cat_cols.append(c)

print(f"[02] #num={len(num_cols)}  #cat={len(cat_cols)}  #text={len(txt_cols)}")


[02] #num=8  #cat=3  #text=0


In [9]:

# ==== Build structured (num + cat) transformer ====
# Note: keep sparse to allow hstack with TF-IDF later.
structured = None
if len(num_cols) + len(cat_cols) > 0:
    transformers = []
    if len(num_cols) > 0:
        transformers.append(("num", StandardScaler(with_mean=False), num_cols))
    if len(cat_cols) > 0:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols))
    structured = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=1.0)
else:
    print("[02] No numeric/categorical columns detected for structured features.")


In [10]:

# ==== Text pipeline (optional) ====
tfidf = None
MAX_TFIDF = int(globals().get("MAX_TFIDF", 50000))

def _join_text_cols(df, cols):
    if len(cols)==0:
        return None
    Xtxt = df[cols].astype(str).fillna("")
    # Concatenate columns with space
    return Xtxt.apply(lambda r: " ".join([v for v in r.values if v and v!="nan"]), axis=1).values

if len(txt_cols) > 0:
    tfidf = TfidfVectorizer(max_features=MAX_TFIDF, ngram_range=(1,2))
    print(f"[02] TF-IDF configured with max_features={MAX_TFIDF}")
else:
    print("[02] No text columns detected; will skip TF-IDF.")


[02] No text columns detected; will skip TF-IDF.


In [11]:

# ==== Fit & transform ====
def _fit_transform_structured(df_tr, df_va, df_te):
    if structured is None:
        return None, None, None, None
    Xt = structured.fit_transform(df_tr)
    Xv = structured.transform(df_va)
    Xs = structured.transform(df_te)
    return structured, Xt, Xv, Xs

def _fit_transform_text(df_tr, df_va, df_te):
    if tfidf is None:
        return None, None, None, None
    tr_txt = _join_text_cols(df_tr, txt_cols)
    va_txt = _join_text_cols(df_va, txt_cols)
    te_txt = _join_text_cols(df_te, txt_cols)
    if tr_txt is None:
        return None, None, None, None
    Tt = tfidf.fit_transform(tr_txt)
    Tv = tfidf.transform(va_txt)
    Ts = tfidf.transform(te_txt)
    return tfidf, Tt, Tv, Ts

# Structured
struct_model, X_tr_struct, X_va_struct, X_te_struct = _fit_transform_structured(
    train_df, val_df, test_df
)

# Text
tfidf_model, X_tr_txt, X_va_txt, X_te_txt = _fit_transform_text(
    train_df, val_df, test_df
)

def _hstack(a, b):
    if a is None and b is None:
        return None
    if a is None:
        return b.tocsr() if sp.issparse(b) else sp.csr_matrix(b)
    if b is None:
        return a.tocsr() if sp.issparse(a) else sp.csr_matrix(a)
    return sp.hstack([a, b]).tocsr()

X_tr_all = _hstack(X_tr_struct, X_tr_txt)
X_va_all = _hstack(X_va_struct, X_va_txt)
X_te_all = _hstack(X_te_struct, X_te_txt)

# Fallback: if we ended up with 0 features, inject a constant column
def _ensure_nonzero_feats(X, n):
    if X is None or (sp.issparse(X) and X.shape[1]==0) or (isinstance(X, np.ndarray) and X.shape[1]==0):
        print(f"[02] WARNING: Produced 0 features for a split; injecting constant feature ({n}).")
        return sp.csr_matrix(np.ones((n,1), dtype=np.float32))
    return X

X_tr_all = _ensure_nonzero_feats(X_tr_all, len(train_df))
X_va_all = _ensure_nonzero_feats(X_va_all, len(val_df))
X_te_all = _ensure_nonzero_feats(X_te_all, len(test_df))

# Structured-only arrays (could be None); if None, use all as structured
def _structured_or_all(X_struct, X_all, n):
    if X_struct is None:
        return X_all
    return _ensure_nonzero_feats(X_struct, n)

X_tr_struct = _structured_or_all(X_tr_struct, X_tr_all, len(train_df))
X_va_struct = _structured_or_all(X_va_struct, X_va_all, len(val_df))
X_te_struct = _structured_or_all(X_te_struct, X_te_all, len(test_df))

print("[02] Shapes:")
print("  structured: ", X_tr_struct.shape, X_va_struct.shape, X_te_struct.shape)
print("  all:        ", X_tr_all.shape, X_va_all.shape, X_te_all.shape)


[02] Shapes:
  structured:  (36000, 23) (12000, 23) (12000, 23)
  all:         (36000, 23) (12000, 23) (12000, 23)


In [12]:

# ==== Persist arrays ====
def _np_save(path, arr):
    # Save CSR as object npy to be loaded later with allow_pickle=True
    if sp.issparse(arr):
        np.save(path, np.array(arr, dtype=object), allow_pickle=True)
    else:
        np.save(path, arr, allow_pickle=True)

# y arrays
y_train = train_df["__target__"].to_numpy(dtype=int).ravel()
y_val   = val_df["__target__"].to_numpy(dtype=int).ravel()
y_test  = test_df["__target__"].to_numpy(dtype=int).ravel()

_np_save(SPLIT_DIR / "X_train.npy", X_tr_struct)
_np_save(SPLIT_DIR / "X_val.npy",   X_va_struct)
_np_save(SPLIT_DIR / "X_test.npy",  X_te_struct)

_np_save(SPLIT_DIR / "X_train_all.npy", X_tr_all)
_np_save(SPLIT_DIR / "X_val_all.npy",   X_va_all)
_np_save(SPLIT_DIR / "X_test_all.npy",  X_te_all)

np.save(SPLIT_DIR / "y_train.npy", y_train, allow_pickle=True)
np.save(SPLIT_DIR / "y_val.npy",   y_val,   allow_pickle=True)
np.save(SPLIT_DIR / "y_test.npy",  y_test,  allow_pickle=True)

print("[02] Saved staged arrays.")


[02] Saved staged arrays.


In [13]:

# ==== Save metadata ====
meta = {
    "random_state": RANDOM_STATE,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "target_col_original": TARGET_COL,
    "target_col_final": "__target__",
    "n_rows": int(len(data)),
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "n_test": int(len(y_test)),
    "class_counts": {
        "train": {int(k): int(v) for k,v in zip(*np.unique(y_train, return_counts=True))},
        "val":   {int(k): int(v) for k,v in zip(*np.unique(y_val, return_counts=True))},
        "test":  {int(k): int(v) for k,v in zip(*np.unique(y_test, return_counts=True))},
    },
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "txt_cols": txt_cols,
    "shapes": {
        "structured": {
            "train": list(map(int, list(sp.csr_matrix(X_tr_struct).shape))),
            "val":   list(map(int, list(sp.csr_matrix(X_va_struct).shape))),
            "test":  list(map(int, list(sp.csr_matrix(X_te_struct).shape))),
        },
        "all": {
            "train": list(map(int, list(sp.csr_matrix(X_tr_all).shape))),
            "val":   list(map(int, list(sp.csr_matrix(X_va_all).shape))),
            "test":  list(map(int, list(sp.csr_matrix(X_te_all).shape))),
        }
    }
}

with open(SPLIT_DIR / "02_feature_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("[02] Wrote metadata →", SPLIT_DIR / "02_feature_meta.json")


[02] Wrote metadata → out/split_preproc/02_feature_meta.json


## Section 2.1 — Payload Sequence (TF-IDF)

**What this does**
- Fold `payload_byte_*` → a single `payload` column (space-separated uppercase hex tokens)
- Clean/sanitise payload strings; configure and fit a TF-IDF vectoriser (byte-token n-grams)
- Transform validation/test splits; assert shapes; persist artefacts and MANIFEST

**Why**
- Provide stable, reproducible payload features for modelling while avoiding leakage
- Keep pipeline resume-safe with explicit artefact persistence under `staging/section_2/`


In [14]:
# ======================================================
# Section 2.1 — Payload Sequence (TF-IDF)
#   • Fold payload_byte_* → 'payload' (uppercase hex tokens)
#   • Sanitise, vectorise (TF-IDF), persist, and validate
#   • Soft-skip when no usable frames found
# ======================================================
print(">>> Section 2.1: Payload Sequence (TF-IDF)")

import re, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse as sp
import joblib

# ---------- Helpers: fold payload_byte_* into 'payload'
def _find_payload_cols(df):
    pat = re.compile(r"^payload_byte_(\d+)$")
    cols = []
    for c in df.columns:
        m = pat.match(c)
        if m:
            cols.append((int(m.group(1)), c))
    cols.sort(key=lambda x: x[0])
    return [c for _, c in cols]

def _fold_bytes_row(row, cols):
    vals = []
    for c in cols:
        v = row.get(c)
        if pd.isna(v):
            vals.append("00")
        else:
            try:
                iv = int(v)
                if iv < 0: iv = 0
                if iv > 255: iv = 255
                vals.append(f"{iv:02X}")
            except Exception:
                s = str(v).strip().upper()
                vals.append((s[:2] if s else "00"))
    return " ".join(vals)

def fold_payload(df):
    cols = _find_payload_cols(df)
    if not cols:
        # If neither payload_byte_* nor payload present, return unchanged and let caller decide
        return df
    payload_series = df[cols].apply(lambda r: _fold_bytes_row(r, cols), axis=1)
    out = df.drop(columns=cols, inplace=False)
    out['payload'] = payload_series.astype(str)
    return out

# ---------- Apply folding + sanitisation (soft-skip when not applicable)
def sanitise_payload_col(df):
    assert 'payload' in df.columns, "'payload' column missing; run folding first"
    s = df['payload'].astype(str).str.strip()
    s = s.replace({'': '00'}).str.replace(r"\s+", " ", regex=True)
    df = df.copy(); df['payload'] = s
    return df

targets = []

def try_fold_and_sanitise(name):
    g = globals()
    if name in g:
        frame = g[name]
        frame2 = fold_payload(frame)
        # proceed only if payload present after folding or already existed
        if 'payload' in frame2.columns:
            frame2 = sanitise_payload_col(frame2)
            g[name] = frame2
            targets.append(name)

for _name in ['df','X_train','X_val','X_test']:
    try_fold_and_sanitise(_name)

if not targets:
    SECTION21_SKIPPED = True
    print("[2.1] No frames had payload_byte_* or existing 'payload' — skipping TF-IDF setup.")
else:
    SECTION21_SKIPPED = False
    print(f"[2.1] Folding+sanitisation applied to: {', '.join(targets)}")

    # ---------- TF-IDF configuration (byte tokens as 'words')
    TFIDF_ANALYZER = 'word'
    TFIDF_TOKEN_PATTERN = r"\S+"   # tokens separated by spaces: '00', 'AF', ...
    TFIDF_NGRAM_RANGE = (1, 3)
    TFIDF_MAX_FEATURES = 50000
    TFIDF_MIN_DF = 2
    TFIDF_NORM = 'l2'

    vectorizer = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        token_pattern=TFIDF_TOKEN_PATTERN,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        lowercase=False,
        dtype=np.float32,
        norm=TFIDF_NORM,
    )
    print("[2.1] Vectorizer configured.")

    # ---------- Fit and persist
    stage_root = Path(STAGE_ROOT) if 'STAGE_ROOT' in globals() else Path('staging')
    sec_dir = stage_root / 'section_2'
    sec_dir.mkdir(parents=True, exist_ok=True)

    if 'X_train' in targets:
        payload_train = X_train['payload']
    elif 'df' in targets:
        payload_train = df['payload']
    else:
        raise AssertionError('Usable frames exist but none suitable for fitting (need X_train or df)')

    X_train_payload = vectorizer.fit_transform(payload_train.astype(str).tolist())
    joblib.dump(vectorizer, sec_dir / 'vectorizer.joblib')
    print(f"[2.1] Fitted vectorizer. Train TF-IDF shape: {X_train_payload.shape}")

    # ---------- Transform validation/test
    if 'X_val' in targets:
        X_val_payload = vectorizer.transform(X_val['payload'].astype(str).tolist())
        assert sp.issparse(X_val_payload)
        print(f"[2.1] Val TF-IDF shape: {X_val_payload.shape}")
    if 'X_test' in targets:
        X_test_payload = vectorizer.transform(X_test['payload'].astype(str).tolist())
        assert sp.issparse(X_test_payload)
        print(f"[2.1] Test TF-IDF shape: {X_test_payload.shape}")

    # ---------- Assertions & manifest
    assert X_train_payload.shape[0] > 0 and X_train_payload.shape[1] > 0, "Empty train matrix"
    observed_vocab = len(vectorizer.vocabulary_)
    assert observed_vocab <= TFIDF_MAX_FEATURES, "Observed vocab exceeds configured max_features"

    manifest = {
        'section': '2.1',
        'vectorizer': {
            'path': str((sec_dir / 'vectorizer.joblib').resolve()),
            'ngram_range': TFIDF_NGRAM_RANGE,
            'max_features': TFIDF_MAX_FEATURES,
            'min_df': TFIDF_MIN_DF,
            'analyzer': TFIDF_ANALYZER,
            'norm': TFIDF_NORM,
        },
        'shapes': {
            'train': tuple(map(int, X_train_payload.shape)),
            'val': tuple(map(int, X_val_payload.shape)) if 'X_val_payload' in globals() else None,
            'test': tuple(map(int, X_test_payload.shape)) if 'X_test_payload' in globals() else None,
        }
    }
    with open(sec_dir / 'MANIFEST.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"[2.1] MANIFEST written to {sec_dir}")


>>> Section 2.1: Payload Sequence (TF-IDF)
[2.1] No frames had payload_byte_* or existing 'payload' — skipping TF-IDF setup.
